# 04. Scoreboard (Image × Regression, ADR-019)

DSC ↔ probe 성능(주) 상관 + finetune spot-check(probe/DSC vs finetune) + §2 가중치 선정.

In [1]:
from google.colab import drive; drive.mount('/content/drive')
import os, sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from scipy.optimize import minimize
BASE = '/content/drive/MyDrive/capstone/dsc'; RESULTS_DIR = f'{BASE}/results'
CHARTS = f'{RESULTS_DIR}/charts_image_regression'; os.makedirs(CHARTS, exist_ok=True)
if BASE not in sys.path: sys.path.insert(0, BASE)
dsc = pd.read_csv(f'{RESULTS_DIR}/dsc_scores_image_regression.csv')
perf = pd.read_csv(f'{RESULTS_DIR}/model_performance_image_regression.csv')
probe = perf[perf.method == 'probe'].copy()
# config별 probe 평균 R² (주 성능 지표)
probe_mean = probe.groupby(['dataset','polluter','level'])['score'].mean().reset_index(name='probe_R2')
m = probe_mean.merge(dsc[['dataset','polluter','level','score','grade']], on=['dataset','polluter','level']).rename(columns={'score':'dsc_score'})
print(f'probe행 {len(probe)}, config {len(probe_mean)}, merged {len(m)}')

Mounted at /content/drive
probe행 208, config 52, merged 52


In [2]:
# 1. DSC ↔ probe 성능 상관 (dataset별 = 합격 단위)
THR = 0.40
print('[A] dataset별 r(DSC, probe평균R2)')
verdict = {}
for ds, sub in m.groupby('dataset'):
    r, p = pearsonr(sub['dsc_score'], sub['probe_R2']); rs, _ = spearmanr(sub['dsc_score'], sub['probe_R2'])
    verdict[ds] = r >= THR
    print(f'  {ds:14s} n={len(sub):2d} r={r:+.4f} p={p:.1e} rho={rs:+.4f} [{"PASS" if r>=THR else "FAIL"}]')
# 모델별(probe 4종) r
print('[B] probe 모델별 r')
mm = probe.rename(columns={'score':'probe_score'}).merge(
    dsc[['dataset','polluter','level','score']].rename(columns={'score':'dsc_score'}),
    on=['dataset','polluter','level'])
for (ds, mdl), sub in mm.groupby(['dataset','model']):
    if len(sub) < 3 or sub['probe_score'].std()==0: continue
    r,_ = pearsonr(sub['dsc_score'], sub['probe_score'])
    print(f'  {ds:14s} {mdl:14s} n={len(sub):2d} r={r:+.4f}')
plt.figure(figsize=(9,5))
for ds, sub in m.groupby('dataset'): plt.scatter(sub['dsc_score'], sub['probe_R2'], label=ds, alpha=.6)
plt.xlabel('DSC'); plt.ylabel('probe mean R2'); plt.legend(); plt.grid(alpha=.3)
plt.savefig(f'{CHARTS}/01_scatter.png', dpi=150); plt.show()

[A] dataset별 r(DSC, probe평균R2)
  SCUT_FBP5500   n=26 r=+0.6680 p=1.9e-04 rho=+0.7479 [PASS]
  UTKFace        n=26 r=+0.4929 p=1.1e-02 rho=+0.5317 [PASS]
[B] probe 모델별 r


KeyError: 'score'

In [ ]:
# 2. finetune spot-check: probe/DSC가 실제 finetune 성능을 따라가나
ft = perf[perf.method == 'finetune']
if len(ft):
    fm = ft.merge(probe_mean, on=['dataset','polluter','level']).merge(
        dsc[['dataset','polluter','level','score']], on=['dataset','polluter','level']).rename(columns={'score':'dsc_score'})
    print(f'spot-check {len(fm)}건 (blur×levels+baseline)')
    if len(fm) >= 3:
        r1,_ = pearsonr(fm['dsc_score'], fm['score']); r2,_ = pearsonr(fm['probe_R2'], fm['score'])
        print(f'  r(DSC, finetune R2)   = {r1:+.4f}')
        print(f'  r(probe R2, finetune) = {r2:+.4f}  ← proxy 타당성 (양수면 probe가 실제 학습 성능 추종)')
        for _,row in fm.iterrows():
            print(f'    {row.dataset}/{row.polluter}_{int(row.level*100)}: DSC={row.dsc_score:.1f} probe={row.probe_R2:.3f} finetune={row.score:.3f}')
else:
    print('finetune spot-check 결과 없음 (03 미실행)')

In [ ]:
# 3. §2 제약 최적화 가중치 (튜닝 UTKFace / held-out SCUT) — 주 성능=probe 평균 R²
from dsc_framework import DEFAULT_WEIGHTS_IMAGE_REG
METRICS = list(DEFAULT_WEIGHTS_IMAGE_REG.keys()); DEFAULT_W = dict(DEFAULT_WEIGHTS_IMAGE_REG)
mf = probe_mean.merge(dsc[['dataset','polluter','level']+METRICS], on=['dataset','polluter','level'])
tune = mf[mf.dataset=='UTKFace']; held = mf[mf.dataset=='SCUT_FBP5500']
print(f'tune n={len(tune)}, held n={len(held)}')
def rw(X,y,w):
    s=X@w; return pearsonr(s,y)[0] if s.std()>1e-12 else 0.0
Xt,yt = tune[METRICS].values, tune['probe_R2'].values
Xh,yh = held[METRICS].values, held['probe_R2'].values
wd = np.array([DEFAULT_W[k] for k in METRICS])
print(f'[Default] tune r={rw(Xt,yt,wd):+.4f} held r={rw(Xh,yh,wd):+.4f}')
WEAK={'outlier_ratio','validity','feature_correlation'}; CORE={'completeness_image','target_smoothness'}
bnds=[(0.02,0.15) if k in WEAK else (0.02,0.35) for k in METRICS]
cons=[{'type':'eq','fun':lambda w:w.sum()-1.0}]
for j,k in enumerate(METRICS):
    if k in CORE: cons.append({'type':'ineq','fun':(lambda w,j=j:w[j]-0.10)})
best=None
for sd in range(20):
    w0=np.random.RandomState(sd).dirichlet(np.ones(len(METRICS)))
    r=minimize(lambda w:-rw(Xt,yt,w), w0, method='SLSQP', bounds=bnds, constraints=cons, options={'maxiter':600,'ftol':1e-9})
    if best is None or r.fun<best.fun: best=r
wo=np.clip(best.x,0,None); wo/=wo.sum()
print(f'[제약최적] tune r={rw(Xt,yt,wo):+.4f} held r={rw(Xh,yh,wo):+.4f}')
print('top5:', ', '.join(f'{k}={w:.2f}' for k,w in sorted(zip(METRICS,wo),key=lambda x:-x[1])[:5]))
json.dump({'weights':{k:float(w) for k,w in zip(METRICS,wo)},'tune':'UTKFace','held_out':'SCUT_FBP5500',
           'perf':'probe_mean_R2','r_default_held':float(rw(Xh,yh,wd)),'r_opt_held':float(rw(Xh,yh,wo))},
          open(f'{RESULTS_DIR}/tuned_weights_image_regression.json','w',encoding='utf-8'), ensure_ascii=False, indent=2)
print('저장: tuned_weights_image_regression.json'); print('--- 04 완료 ---')